# Recommendation Systems - Complete Usage Examples

This notebook demonstrates how to use Content-Based, Collaborative Filtering, and Hybrid recommendation models.

In [1]:
import pandas as pd
import numpy as np
from src.data_processing.data_loader import MovieLensDataLoader
from src.models.content_based import ContentBasedRecommender, ContentBasedConfig
from src.models.collaborative_filtering import CollaborativeFiltering
from src.models.hybrid import HybridRecommender
import warnings
warnings.filterwarnings('ignore')

## 1. Load and Prepare Data

In [2]:
loader = MovieLensDataLoader()
data_dict = loader.load_data()
await loader.letterboxd_data_async(max_concurrent_requests=500)

movies_df = pd.DataFrame(loader.movie_data)
ratings_df = data_dict['ratings']
genre_features = loader.preprocess_movies()
movies_df = pd.concat([movies_df, genre_features], axis=1)
movies_df = movies_df.dropna().reset_index(drop=True)

print(f"Movies: {len(movies_df)}")
print(f"Ratings: {len(ratings_df)}")
print(f"Users: {ratings_df['userId'].nunique()}")

INFO:src.data_processing.data_loader:Loading MovieLens dataset...
INFO:src.data_processing.data_loader:Loading existing data from cache: data/processed/movie_metadata.parquet
INFO:src.data_processing.data_loader:Found 113 missing movies with valid TMDB IDs. Fetching...
INFO:src.data_processing.data_loader:DOWNLOAD SUMMARY: Requested: 113 | Saved: 0 | Failed: 113


Movies: 9589
Ratings: 100836
Users: 610


## 2. Content-Based Filtering

In [3]:
config = ContentBasedConfig(
    main_actor_weight=0.3,
    director_weight=0.3,
    cast_weight=0.3,
    keywords_weight=0.6,
    numerical_weight=0.1,
    similarity_threshold=0.15,
    top_k_default=10
)

cb_model = ContentBasedRecommender(config=config)
cb_model.fit(movies_df=movies_df, ratings_df=ratings_df)
print("Content-Based model trained successfully!")

2026-06-30 11:48:17.545 | INFO     | src.models.content_based:fit:618 - Starting model fitting
2026-06-30 11:48:17.550 | INFO     | src.models.content_based:fit:625 - Cleaning text columns
2026-06-30 11:48:17.612 | INFO     | src.models.content_based:fit:629 - Building cast and keyword strings
2026-06-30 11:48:17.613 | INFO     | src.models.content_based:fit:631 - Processing cast entries
Building keyword strings: 100%|██████████| 9589/9589 [00:00<00:00, 451631.96it/s]
2026-06-30 11:48:17.670 | INFO     | src.models.content_based:fit:638 - Preprocessing numerical and rating features
2026-06-30 11:48:17.671 | INFO     | src.models.content_based:_preprocess_numerical_features:124 - Preprocessing numerical features
2026-06-30 11:48:17.680 | DEBUG    | src.models.content_based:_fit_scaler:110 - Clipping runtime between 25.0 and 178.0
2026-06-30 11:48:17.705 | INFO     | src.models.content_based:_preprocess_actor_director_ratings:141 - Computing actor and director average ratings
2026-06-30 

Content-Based model trained successfully!


### 2.1 Get Recommendations for a User

In [4]:
user_id = 444
user_history = ratings_df[ratings_df['userId'] == user_id]
watched_items = set(user_history['movieId'].values)
liked_items = set(user_history[user_history['rating'] >= 4.0]['movieId'].values)

print(f"User {user_id} has watched {len(watched_items)} movies")
print(f"User {user_id} liked {len(liked_items)} movies (rating >= 4.0)")

User 444 has watched 42 movies
User 444 liked 27 movies (rating >= 4.0)


In [5]:
recommendations = cb_model.get_top_k_with_titles(
    user_id=user_id,
    watched_items=watched_items,
    k=10
)

print(f"\nTop 10 Recommendations for User {user_id}:")
print("=" * 80)
for i, rec in enumerate(recommendations, 1):
    print(f"{i:2}. {rec['title']:<50} (Score: {rec['score']:.4f})")


Top 10 Recommendations for User 444:
 1. GoodFellas                                         (Score: 0.7444)
 2. Saving Private Ryan                                (Score: 0.7338)
 3. The Terminator                                     (Score: 0.7336)
 4. The Godfather Part II                              (Score: 0.7300)
 5. Aliens                                             (Score: 0.7299)
 6. X-Men                                              (Score: 0.7276)
 7. Back to the Future                                 (Score: 0.7272)
 8. Titanic                                            (Score: 0.7272)
 9. Avatar                                             (Score: 0.7265)
10. Star Wars: Episode I - The Phantom Menace          (Score: 0.7262)


### 2.2 Explain Recommendations

In [6]:
movie_id_to_title = dict(zip(movies_df['movieId'], movies_df['title']))

if recommendations:
    rec_movie_id = recommendations[0]['movieId']
    rec_title = recommendations[0]['title']
    
    print(f"Why was '{rec_title}' recommended?")
    print("=" * 80)
    
    reasons = cb_model.explain_recommendation(
        movie_id=rec_movie_id,
        liked_items=liked_items,
        top_n_reasons=5
    )
    
    for i, reason in enumerate(reasons, 1):
        reason_title = movie_id_to_title.get(reason['movie_id'], f"ID: {reason['movie_id']}")
        print(f"{i}. {reason_title} (Similarity: {reason['similarity']:.4f})")

Why was 'GoodFellas' recommended?
1. Casino (Similarity: 0.9996)
2. Taxi Driver (Similarity: 0.9920)


### 2.3 Find Similar Movies

In [7]:
target_movie_id = 4.0
target_movie_title = movies_df[movies_df['movieId'] == target_movie_id]['title'].values[0]

print(f"Movies similar to '{target_movie_title}':")
print("=" * 80)

similar_movies = cb_model.get_similar_movies(
    movie_id=target_movie_id,
    top_n=10
)

for i, movie in enumerate(similar_movies, 1):
    title = movie_id_to_title.get(movie['movie_id'], f"ID: {movie['movie_id']}")
    print(f"{i:2}. {title:<50} (Similarity: {movie['similarity']:.4f})")

Movies similar to 'Waiting to Exhale':
 1. First Daughter                                     (Similarity: 0.8498)
 2. Hope Floats                                        (Similarity: 0.8376)
 3. You Got Served                                     (Similarity: 0.7587)
 4. Zodiac                                             (Similarity: 0.7353)
 5. Dil To Pagal Hai                                   (Similarity: 0.7352)
 6. The Chronicles of Narnia: The Lion, the Witch and the Wardrobe (Similarity: 0.7352)
 7. The Matrix Reloaded                                (Similarity: 0.7351)
 8. Armageddon                                         (Similarity: 0.7351)
 9. 49 Up                                              (Similarity: 0.7351)
10. After Life                                         (Similarity: 0.7351)


### 2.4 Predict Scores for Specific Movies

In [8]:
candidate_movies = [1.0, 2.0, 3.0, 4.0, 5.0]
scores = cb_model.predict_scores(user_id=user_id, item_ids=candidate_movies)

print(f"Predicted scores for User {user_id}:")
print("=" * 80)
for movie_id, score in zip(candidate_movies, scores):
    title = movie_id_to_title.get(movie_id, f"ID: {movie_id}")
    print(f"{title:<50} -> Score: {score:.4f}")

Predicted scores for User 444:
Toy Story                                          -> Score: 0.7062
Jumanji                                            -> Score: 0.7016
Grumpier Old Men                                   -> Score: 0.7042
Waiting to Exhale                                  -> Score: 0.6918
Father of the Bride Part II                        -> Score: 0.7071


## 3. Collaborative Filtering

In [9]:
cf_model = CollaborativeFiltering(
    k_components=50,
    random_state=42
)
cf_model.fit(df_ratings=ratings_df)
print("Collaborative Filtering model trained successfully!")

2026-06-30 11:48:24.067 | INFO     | src.models.collaborative_filtering:__init__:99 - CollaborativeFiltering initialised. Explicit-feedback biased ALS model ready for training.
ALS epochs: 100%|██████████| 20/20 [00:01<00:00, 18.88it/s]
--- Step: Training complete ---
  Wall clock time : 1.16 s
  CPU time (user) : 11.07 s
  Memory RSS      : 389.8 MB
  Memory VMS      : 6307.2 MB
  CPU usage       : 0.0 %
2026-06-30 11:48:25.228 | INFO     | src.models.collaborative_filtering:fit:190 - Fitting complete | wall=1.16s cpu=11.08s mem=389.8MB


Collaborative Filtering model trained successfully!


### 3.1 Get Recommendations

In [10]:
cf_recommendations = cf_model.get_top_k_recommendations(
    user_id=user_id,
    watched_items=watched_items,
    k=10
)

print(f"\nTop 10 CF Recommendations for User {user_id}:")
print("=" * 80)
for i, movie_id in enumerate(cf_recommendations, 1):
    title = movie_id_to_title.get(movie_id, f"ID: {movie_id}")
    score = cf_model.predict_score(user_id, movie_id)
    print(f"{i:2}. {title:<50} (Predicted Rating: {score:.4f})")


Top 10 CF Recommendations for User 444:
 1. The King's Speech                                  (Predicted Rating: 5.0000)
 2. The Royal Tenenbaums                               (Predicted Rating: 5.0000)
 3. The Social Network                                 (Predicted Rating: 5.0000)
 4. Deadpool                                           (Predicted Rating: 5.0000)
 5. Planet of the Apes                                 (Predicted Rating: 5.0000)
 6. Contact                                            (Predicted Rating: 5.0000)
 7. Anchorman: The Legend of Ron Burgundy              (Predicted Rating: 5.0000)
 8. The Talented Mr. Ripley                            (Predicted Rating: 5.0000)
 9. The Avengers                                       (Predicted Rating: 5.0000)
10. Jackie Brown                                       (Predicted Rating: 5.0000)


### 3.2 Explain CF Recommendations

In [11]:
if cf_recommendations:
    cf_movie_id = cf_recommendations[0]
    cf_title = movie_id_to_title.get(cf_movie_id, f"ID: {cf_movie_id}")
    
    print(f"Why was '{cf_title}' recommended (CF)?")
    print("=" * 80)
    
    cf_reasons = cf_model.explain_recommendation(
        movie_id=cf_movie_id,
        liked_items=liked_items,
        top_n_reasons=5
    )
    
    for i, reason in enumerate(cf_reasons, 1):
        reason_title = movie_id_to_title.get(reason['movie_id'], f"ID: {reason['movie_id']}")
        print(f"{i}. {reason_title} (Similarity: {reason['similarity']:.4f})")

Why was 'The King's Speech' recommended (CF)?
1. The Usual Suspects (Similarity: 0.3457)
2. La Haine (Similarity: 0.3230)
3. Beauty and the Beast (Similarity: 0.3110)
4. The Shawshank Redemption (Similarity: 0.3003)
5. Pulp Fiction (Similarity: 0.2720)


## 4. Hybrid Model

In [12]:
hybrid_model = HybridRecommender(
    cf_model=cf_model,
    cb_model=cb_model,
    alpha=0.8
)

hybrid_model.fit(movies_df=movies_df, ratings_df=ratings_df)
print("Hybrid model trained successfully!")

2026-06-30 11:48:25.271 | INFO     | src.models.content_based:fit:618 - Starting model fitting
2026-06-30 11:48:25.277 | INFO     | src.models.content_based:fit:625 - Cleaning text columns
2026-06-30 11:48:25.325 | INFO     | src.models.content_based:fit:629 - Building cast and keyword strings
2026-06-30 11:48:25.333 | INFO     | src.models.content_based:fit:631 - Processing cast entries
Building keyword strings: 100%|██████████| 9589/9589 [00:00<00:00, 509632.54it/s]
2026-06-30 11:48:25.466 | INFO     | src.models.content_based:fit:638 - Preprocessing numerical and rating features
2026-06-30 11:48:25.487 | INFO     | src.models.content_based:_preprocess_numerical_features:124 - Preprocessing numerical features
2026-06-30 11:48:25.492 | DEBUG    | src.models.content_based:_fit_scaler:110 - Clipping runtime between 25.0 and 178.0
2026-06-30 11:48:25.507 | INFO     | src.models.content_based:_preprocess_actor_director_ratings:141 - Computing actor and director average ratings
ALS epochs:

Hybrid model trained successfully!


### 4.1 Get Hybrid Recommendations

In [13]:
hybrid_recommendations = hybrid_model.get_top_k_recommendations(
    user_id=user_id,
    watched_items=watched_items,
    k=10
)

print(f"\nTop 10 Hybrid Recommendations for User {user_id} (alpha=0.5):")
print("=" * 80)
for i, movie_id in enumerate(hybrid_recommendations, 1):
    title = movie_id_to_title.get(movie_id, f"ID: {movie_id}")
    score = hybrid_model.predict_scores(user_id, [movie_id])[0]
    print(f"{i:2}. {title:<50} (Score: {score:.4f})")


Top 10 Hybrid Recommendations for User 444 (alpha=0.5):
 1. Eraser                                             (Score: 4.1449)
 2. Crimson Tide                                       (Score: 4.1444)
 3. The Abyss                                          (Score: 4.1441)
 4. Casino Royale                                      (Score: 4.1441)
 5. A Beautiful Mind                                   (Score: 4.1439)
 6. Reservoir Dogs                                     (Score: 4.1438)
 7. Contact                                            (Score: 4.1438)
 8. Inception                                          (Score: 4.1437)
 9. The Avengers                                       (Score: 4.1437)
10. The Lord of the Rings: The Fellowship of the Ring  (Score: 4.1436)


### 4.2 Explain Hybrid Recommendations

In [14]:
if hybrid_recommendations:
    hybrid_movie_id = hybrid_recommendations[0]
    hybrid_title = movie_id_to_title.get(hybrid_movie_id, f"ID: {hybrid_movie_id}")
    
    print(f"Why was '{hybrid_title}' recommended (Hybrid)?")
    print("=" * 80)
    
    hybrid_reasons = hybrid_model.explain_recommendation(
        movie_id=hybrid_movie_id,
        liked_items=liked_items,
        top_n_reasons=5
    )
    
    for i, reason in enumerate(hybrid_reasons, 1):
        reason_title = movie_id_to_title.get(reason['movie_id'], f"ID: {reason['movie_id']}")
        print(f"{i}. {reason_title} (Combined Similarity: {reason['similarity']:.4f})")

Why was 'Eraser' recommended (Hybrid)?
1. The Mask (Combined Similarity: 0.8586)
2. True Lies (Combined Similarity: 0.8353)
3. Terminator 2: Judgment Day (Combined Similarity: 0.8291)
4. The Crow (Combined Similarity: 0.4192)
5. Mary Shelley's Frankenstein (Combined Similarity: 0.4051)


## 5. Compare All Three Models

In [15]:
print(f"\nComparison of Top 5 Recommendations for User {user_id}:")
print("=" * 120)
print(f"{'Rank':<6}{'Content-Based':<35}{'Collaborative':<35}{'Hybrid':<35}")
print("-" * 120)

for i in range(5):
    cb_title = movie_id_to_title.get(recommendations[i]['movieId'], "N/A")[:33] if i < len(recommendations) else "N/A"
    cf_title = movie_id_to_title.get(cf_recommendations[i], "N/A")[:33] if i < len(cf_recommendations) else "N/A"
    hybrid_title = movie_id_to_title.get(hybrid_recommendations[i], "N/A")[:33] if i < len(hybrid_recommendations) else "N/A"
    
    print(f"{i+1:<6}{cb_title:<35}{cf_title:<35}{hybrid_title:<35}")


Comparison of Top 5 Recommendations for User 444:
Rank  Content-Based                      Collaborative                      Hybrid                             
------------------------------------------------------------------------------------------------------------------------
1     GoodFellas                         The King's Speech                  Eraser                             
2     Saving Private Ryan                The Royal Tenenbaums               Crimson Tide                       
3     The Terminator                     The Social Network                 The Abyss                          
4     The Godfather Part II              Deadpool                           Casino Royale                      
5     Aliens                             Planet of the Apes                 A Beautiful Mind                   


## 6. Save and Load Models

In [16]:
cb_model.save_artifacts(
    similarity_path="data/processed/artifacts/cb_similarity.npy",
    mapping_path="data/processed/artifacts/cb_mapping.pkl",
    preprocessors_path="data/processed/artifacts/cb_preprocessors.pkl"
)
print("Content-Based model saved!")

2026-06-30 11:48:29.744 | INFO     | src.models.content_based:save_artifacts:583 - Artifacts saved to data/processed/artifacts


Content-Based model saved!


In [17]:
cb_model_loaded = ContentBasedRecommender(config=config)
cb_model_loaded.load_artifacts(
    similarity_path="data/processed/artifacts/cb_similarity.npy",
    mapping_path="data/processed/artifacts/cb_mapping.pkl",
    preprocessors_path="data/processed/artifacts/cb_preprocessors.pkl"
)
print("Content-Based model loaded!")

test_recommendations = cb_model_loaded.get_top_k_recommendations(
    user_id=user_id,
    watched_items=watched_items,
    k=5
)
print(f"Test: Generated {len(test_recommendations)} recommendations with loaded model")

2026-06-30 11:48:29.756 | INFO     | src.models.content_based:load_artifacts:590 - Loading artifacts from data/processed/artifacts


TypeError: 'numpy.ndarray' object does not support the context manager protocol

## 7. Full User Profile Display

In [18]:
cb_model.show_user_profile_and_recommendations(
    user_id=user_id,
    ratings_df=ratings_df,
    movies_df=movies_df,
    k=10,
    top_rated_count=5,
    reasons_count=3
)


=== User 444 Profile & Recommendations ===

Liked movies (27):
  6 - Heat
  16 - Casino
  21 - Get Shorty
  47 - Se7en
  50 - The Usual Suspects

Top 10 recommendations:
  1213 - GoodFellas (score: 0.7444)


ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()